In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from collections import Counter

In [ ]:
class DecisionTree:
    def __init__(self, max_depth=10, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    def fit(self, X, y):
        self.root = self._grow_tree(X, y)

    def _entropy(self, y):
        hist = np.bincount(y)
        ps = hist / len(y)
        return -np.sum([p * np.log2(p) for p in ps if p > 0])

    def _split(self, X_column, split_thresh):
        left_idxs = np.argwhere(X_column <= split_thresh).flatten()
        right_idxs = np.argwhere(X_column > split_thresh).flatten()
        return left_idxs, right_idxs

    def _information_gain(self, y, X_column, split_thresh):
        parent_entropy = self._entropy(y)
        left_idxs, right_idxs = self._split(X_column, split_thresh)
        if len(left_idxs) == 0 or len(right_idxs) == 0:
            return 0
        n = len(y)
        n_l, n_r = len(left_idxs), len(right_idxs)
        e_l, e_r = self._entropy(y[left_idxs]), self._entropy(y[right_idxs])
        child_entropy = (n_l / n) * e_l + (n_r / n) * e_r
        return parent_entropy - child_entropy

    def _best_criteria(self, X, y, feat_idxs):
        best_gain = -1
        split_idx, split_thresh = None, None
        for feat_idx in feat_idxs:
            X_column = X[:, feat_idx]
            thresholds = np.unique(X_column)
            for threshold in thresholds:
                gain = self._information_gain(y, X_column, threshold)
                if gain > best_gain:
                    best_gain = gain
                    split_idx = feat_idx
                    split_thresh = threshold
        return split_idx, split_thresh

    def _grow_tree(self, X, y, depth=0):
        n_samples, n_features = X.shape
        n_labels = len(np.unique(y))

        # Остановка, если достигнута глубина или чистота узла
        if (depth >= self.max_depth or n_labels == 1 or n_samples < self.min_samples_split):
            leaf_value = Counter(y).most_common(1)[0][0]
            return {'leaf': True, 'value': leaf_value}

        feat_idxs = np.random.choice(n_features, n_features, replace=False)
        best_feat, best_thresh = self._best_criteria(X, y, feat_idxs)

        left_idxs, right_idxs = self._split(X[:, best_feat], best_thresh)
        left = self._grow_tree(X[left_idxs, :], y[left_idxs], depth + 1)
        right = self._grow_tree(X[right_idxs, :], y[right_idxs], depth + 1)
        return {'leaf': False, 'feature': best_feat, 'threshold': best_thresh, 'left': left, 'right': right}

    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _traverse_tree(self, x, node):
        if node['leaf']:
            return node['value']
        if x[node['feature']] <= node['threshold']:
            return self._traverse_tree(x, node['left'])
        return self._traverse_tree(x, node['right'])

In [ ]:
class RandomForest:
    def __init__(self, n_trees=10, max_depth=10, min_samples_split=2):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.trees = []

    def fit(self, X, y):
        self.trees = []
        for _ in range(self.n_trees):
            tree = DecisionTree(max_depth=self.max_depth, min_samples_split=self.min_samples_split)
            # Бутстрап (выборка с возвращением)
            indices = np.random.choice(len(X), len(X), replace=True)
            tree.fit(X[indices], y[indices])
            self.trees.append(tree)

    def predict(self, X):
        tree_preds = np.array([tree.predict(X) for tree in self.trees])
        # Мажоритарное голосование
        return np.array([Counter(tree_preds[:, i]).most_common(1)[0][0] for i in range(X.shape[0])])

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
clf = RandomForest(n_trees=5)
clf.fit(X_train, y_train)

In [ ]:
predictions = clf.predict(X_test)
accuracy = np.sum(predictions == y_test) / len(y_test)

In [ ]:
print(f"Точность (Accuracy): {accuracy * 100:.2f}%")

# Лабораторная работа. Rain in Australia

https://www.kaggle.com/datasets/jsphyg/weather-dataset-rattle-package

Вы когда-нибудь задумывались, стоит ли завтра брать с собой зонт? С помощью этого набора данных вы можете предсказать дождь на следующий день, обучив модели классификации на целевой переменной RainTomorrow.

Содержание
Этот набор данных включает в себя примерно 10 лет ежедневных наблюдений за погодой из многочисленных мест по всей Австралии.

Переменная RainTomorrow является целевой переменной для прогнозирования. Она отвечает на важнейший вопрос: будет ли дождь на следующий день? (Да или Нет).

Вам необходимо выбрать наилучший алгоритм для решения задачи бинарной классификации на данном датасете среди: C4.5, CART, Случайного леса.

5 баллов будут даны тем студентам, чье значение точности модели будет в пределах 10% от наивысшего по группе.

In [ ]:
import pandas as pd

In [ ]:
# Загрузка датасета
au_rain = pd.read_csv('.csv')

In [ ]:
print(au_rain.head())

In [ ]:
# Предобработка данных и разделение на обучающую и тестовую выборки

In [ ]:
# Наиболее близкая модель к C4.5
model1 = DecisionTreeClassifier(criterion='entropy')

In [ ]:
from sklearn.tree import DecisionTreeClassifier
# CART для классификации (использует Gini по умолчанию)
model2 = DecisionTreeClassifier()

In [ ]:
# Случайный лес
from sklearn.ensemble import RandomForestClassifier
model3 = RandomForestClassifier(n_estimators=100) # 100 деревьев